# CodingTheory-01 — Les codes correcteurs d'erreurs comme objet calculable

**Question.** Shannon (série InformationTheory) montre qu'il existe un taux de transmission fiable — la capacité `C` du canal. **Comment l'atteint-on en pratique ?** Avec des **codes** : on encode un message en un mot plus long, de sorte qu'une poignée d'erreurs du canal devient **détectable**, voire **corrigible**, par un simple calcul.

Ce carnet construit, **en numpy pur** (arithmétique GF(2), aucun GPU), les briques que le reste de la série consommera :

- la **distance de Hamming** et la **distance minimale** d'un code ;
- ce que la distance minimale **achète** (détection / correction) ;
- le **code de parité** (le plus simple) ;
- le **code de Hamming (7,4)** avec **décodage par syndrome** ;
- la **borne de Hamming** — et pourquoi le (7,4) est *parfait*.

Tout est **démontré sur des mots réels** et **asserté par le calcul** : chaque affirmation porte une `assert` que le code vérifie.

## Garde-fous d'honnêteté

- **CPU/numpy** : toute l'arithmétique est `mod 2` (GF(2)) sur des tableaux numpy — c'est l'organe natif, pas un simulateur GPU.
- **Codes petits** : les exemples sont des codes **petits** (parité, (7,4)) choisis pour rester lisibles et s'exécuter instantanément ; la théorie (borne de Hamming, théorème de codage) s'étend aux grands codes des carnets 02-03.
- **Chaque chiffre est asserté** : pas de « on vérifie » au sens vague — une `assert` Python prouve l'affirmation, et le carnet s'exécute de bout en bout.

In [1]:
import numpy as np

# ---- Arithmetique GF(2) : l'organe natif ----
def hamming(a: np.ndarray, b: np.ndarray) -> int:
    """Distance de Hamming : nombre de positions ou deux mots de GF(2) different."""
    return int(np.sum((a + b) % 2))

def min_distance(codewords: np.ndarray) -> int:
    """Distance minimale d'un code : min des distances de Hamming entre paires distinctes."""
    dmin = None
    n = codewords.shape[0]
    for i in range(n):
        for j in range(i + 1, n):
            d = hamming(codewords[i], codewords[j])
            dmin = d if dmin is None else min(dmin, d)
    return dmin

print("Helpers GF(2) definis : hamming(), min_distance()")
print("(les deux fonctions ci-dessus sont les seules dependances du carnet)")

Helpers GF(2) definis : hamming(), min_distance()
(les deux fonctions ci-dessus sont les seules dependances du carnet)


### Ce que montre cette sortie

La sortie ne montre aucun résultat : elle **déclare l'outillage**. Le carnet affiche que ses deux seules dépendances de calcul sont `hamming()` et `min_distance()`, et que toute l'arithmétique se fait modulo `2` — en GF(2), l'addition est un « ou exclusif » : un bit plus un bit redonne un bit.

C'est une information de **portée** plutôt que de contenu, et elle vaut mieux qu'une ligne de silence sur trois points.

- **Aucune dépendance cachée.** Le carnet annonce lui-même que ces deux fonctions sont tout ce dont il dispose. Tout ce qui suit est donc vérifiable à l'œil dans les cellules de code, sans aller chercher une implémentation tierce de théorie des codes pour refaire le raisonnement.
- **Un alphabet, pas des nombres.** Tout ce qui suit manipule des bits, et une distance n'a de sens que sur des mots de même longueur. La distance de Hamming est une **comparaison position par position**, jamais une soustraction : c'est pourquoi le calcul passe par `(a + b) % 2` et non par une comparaison de valeurs.
- **Une arithmétique qui rend l'erreur manipulable.** Modulo `2`, un bit basculé par le canal se ramène à une différence ; c'est cette propriété additive qui rendra possible, plus loin, de **résumer** plusieurs bits erronés par une seule valeur — le syndrome. Un alphabet sans structure additive ne le permettrait pas.

Ce que cette sortie **ne** prouve pas : elle ne dit encore rien de la puissance d'un code. Aucune paire de mots n'a été comparée, aucun code n'a été construit. Elle pose l'instrument ; les chiffres qui suivent mesureront l'instrument.
## 1. La distance de Hamming

La **distance de Hamming** entre deux mots est le nombre de positions où ils diffèrent. C'est la mesure de « combien d'erreurs séparent deux mots » — et la brique de base de tout ce qui suit.

In [2]:
# Exemple de base : deux mots de 7 bits qui different en 2 positions (indices 2 et 4)
a = np.array([1, 0, 1, 1, 0, 0, 1])
b = np.array([1, 0, 0, 1, 1, 0, 1])
d = hamming(a, b)
print(f"a            = {a.tolist()}")
print(f"b            = {b.tolist()}")
print(f"distance(a,b)= {d}")
assert d == 2, "a et b different en 2 positions (indices 2 et 4)"
assert hamming(a, a) == 0, "un mot est a distance 0 de lui-meme"
print("OK : 2 positions de divergence (indices 2 et 4)")

a            = [1, 0, 1, 1, 0, 0, 1]
b            = [1, 0, 0, 1, 1, 0, 1]
distance(a,b)= 2
OK : 2 positions de divergence (indices 2 et 4)


### Ce que montre cette sortie

Les deux mots affichés diffèrent **en deux positions seulement** : la distance vaut 2, et les positions en cause sont les indices 2 et 4 — partout ailleurs, les deux mots portent le même bit. La ligne `distance(a,b)= 2` n'est donc pas un résultat, c'est la **définition qui s'applique**. L'assert `hamming(a, a) == 0` vérifie l'une des propriétés attendues d'une distance : l'identité, un mot est à distance 0 de lui-même.

Deux enseignements, l'un sur la méthode, l'autre sur ce qui manque encore.

- **La distance est un compte, pas une probabilité.** Elle vaut 2 parce que deux positions diffèrent ; pour cette paire de mots, l'écart est *petit*. C'est dire une chose sur *ces* deux mots, et rien sur un code.
- **Un mot isolé ne définit pas de code.** Tant qu'un seul mot est donné, il n'y a pas de distance minimale : celle-ci est un minimum sur *toutes* les paires de codeurs distincts. Cette sortie prépare l'instrument qui calculera ce minimum sur un ensemble entier ; l'ensemble lui-même n'existe pas encore.

Le second mot du couple a d'ailleurs été construit pour différer sur deux positions et **seulement** sur deux : c'est le premier cas d'école du carnet, et il sert de témoin à la définition. Le carnet ne mesure donc encore rien — il vérifie que l'instrument compte ce qu'il doit compter.
## 2. La distance minimale d'un code — et ce qu'elle achète

Un **code** est un ensemble de mots valides (les *codeurs*). Sa **distance minimale** `d_min` est la plus petite distance de Hamming entre deux codeurs distincts. Ce nombre **achète** une capacité précise :

- un code de distance minimale `d` **détecte** jusqu'à `d − 1` erreurs ;
- il **corrige** jusqu'à `⌊(d − 1) / 2⌋` erreurs.

Intuition : si deux codeurs sont à distance `d`, un mot qui a glissé de **moins de `d`** positions reste plus proche du codeur d'origine que de tout autre — on peut donc localiser (et corriger) l'erreur.

In [3]:
def detecte(d: int) -> int:
    return d - 1
def corrige(d: int) -> int:
    return (d - 1) // 2

print("ce que la distance minimale ACHETE :")
for d in (2, 3, 4, 5):
    print(f"  d_min = {d}  ->  detecte {detecte(d)} erreur(s), corrige {corrige(d)} erreur(s)")

ce que la distance minimale ACHETE :
  d_min = 2  ->  detecte 1 erreur(s), corrige 0 erreur(s)
  d_min = 3  ->  detecte 2 erreur(s), corrige 1 erreur(s)
  d_min = 4  ->  detecte 3 erreur(s), corrige 1 erreur(s)
  d_min = 5  ->  detecte 4 erreur(s), corrige 2 erreur(s)


### Ce que montre cette sortie

La table est la **promesse géométrique** d'un code, exprimée en quatre cas. Elle ne montre pas un code : elle montre ce qu'un code de distance minimale donnée prometrait.

- `d_min = 2` détecte 1 erreur et n'en corrige 0 : on sait qu'il y en a une, on ne sait pas où.
- `d_min = 3` détecte 2 erreurs et en corrige 1 : c'est le cas du code de Hamming, qui apparaîtra plus loin.
- `d_min = 4` détecte 3 erreurs mais n'en corrige toujours que 1.
- `d_min = 5` détecte 4 erreurs et en corrige 2.

Le motif est net : quand `d_min` est impair, la correction gagne une erreur ; quand `d_min` est pair, elle stagne. La formule `⌊(d − 1) / 2⌋` ne peut alors qu'arrondir vers le bas, et le budget de correction reste fixé à la même valeur d'un `d_min` pair au suivant.

Ce que la sortie **ne** prouve pas, et qu'il faut dire nettement : ces capacités sont **conditionnelles**. Elles supposent que le code possède effectivement la distance minimale annoncée — un code réel peut en avoir une plus faible, et le tableau ne le verrait pas. Elles supposent aussi que la correction soit *atteignable* : la formule dit ce qu'on pourrait faire si l'on savait localiser l'erreur, pas qu'on sache la localiser. C'est tout l'objet de la suite : le code de parité détecte sans localiser, le code de Hamming localise grâce au syndrome.
## 3. Le code de parité — le plus simple

On ajoute un **bit de parité** au message pour que le nombre total de 1 soit **pair**. C'est un code `(n, n−1)` : un bit de redondance pour `n−1` bits de message. Sa distance minimale est **2** → il **détecte 1 erreur** (mais ne peut pas la *localiser* : on sait qu'il y en a une, pas où).

In [4]:
def parity_encode(msg: np.ndarray) -> np.ndarray:
    """Encode : ajoute un bit de parite paire (nombre total de 1 pair)."""
    p = int(np.sum(msg)) % 2
    return np.concatenate([msg, [p]])

def parity_has_error(codeword: np.ndarray) -> bool:
    """Detecte : parite impaire = un nombre impair d'erreurs est survenu."""
    return bool(int(np.sum(codeword)) % 2 == 1)

msg = np.array([1, 0, 1, 1])
enc = parity_encode(msg)
print(f"message  = {msg.tolist()}")
print(f"encode   = {enc.tolist()}   (parite paire : {int(np.sum(enc)) % 2 == 0})")
assert int(np.sum(enc)) % 2 == 0

# 1 erreur (bascule du bit 0) -> parite impaire -> DETECTEE
noisy = enc.copy(); noisy[0] = 1 - noisy[0]
print(f"apres 1 erreur : {noisy.tolist()}  ->  detectee ? {parity_has_error(noisy)}")
assert parity_has_error(noisy) is True
# 2 erreurs -> parite redevient paire -> NON detectee (la limite du code de parite)
noisy2 = enc.copy(); noisy2[0] = 1 - noisy2[0]; noisy2[3] = 1 - noisy2[3]
print(f"apres 2 erreurs : {noisy2.tolist()}  ->  detectee ? {parity_has_error(noisy2)}  (2 = pair, masquee)")
assert parity_has_error(noisy2) is False

parity_code = np.array([parity_encode(np.array([x, y])) for x in (0, 1) for y in (0, 1)])
print("\nles 4 codeurs du code de parite (mots de 3 bits) :")
print(parity_code)
print("distance minimale =", min_distance(parity_code))
assert min_distance(parity_code) == 2, "le code de parite a d_min = 2"

message  = [1, 0, 1, 1]
encode   = [1, 0, 1, 1, 1]   (parite paire : True)
apres 1 erreur : [0, 0, 1, 1, 1]  ->  detectee ? True
apres 2 erreurs : [0, 0, 1, 0, 1]  ->  detectee ? False  (2 = pair, masquee)

les 4 codeurs du code de parite (mots de 3 bits) :
[[0 0 0]
 [0 1 1]
 [1 0 1]
 [1 1 0]]
distance minimale = 2


### Ce que montre cette sortie

Trois faits dans cette sortie, et le troisième est la limite du code.

- **L'encodage n'ajoute qu'un bit.** Le message `[1, 0, 1, 1]` devient `[1, 0, 1, 1, 1]` : le bit ajouté est celui qui rend pair le nombre total de `1`. C'est une redondance d'**un seul bit**. La sortie présente ensuite le même code de parité sur des messages de deux bits : les quatre messages possibles, chacun muni de son bit de parité, donnent quatre codeurs de trois bits. C'est sur ces quatre mots que la distance minimale est calculée, et elle vaut 2. Le témoin `[1, 0, 1, 1]` est lui un message de quatre bits : il montre l'encodage, il ne fait pas partie de cet ensemble.
- **Une erreur est visible.** Après le retournement du premier bit, le mot reçu `[0, 0, 1, 1, 1]` porte un nombre impair de `1` : la parité est rompue, et le calcul le signale. Le code **détecte**.
- **Deux erreurs sont invisibles.** Après deux retournements, le mot `[0, 0, 1, 0, 1]` porte de nouveau un nombre pair de `1` : la parité est rétablie et le calcul ne voit rien. Le code **ne détecte pas**, et l'assert le confirme.

Ce dernier point est le cœur de la sortie, et il explique pourquoi ce code ne peut pas corriger. La parité ne donne qu'un seul renseignement, « pair ou impair » : une erreur sur une position ou sur une autre produit le même indice. Deux bits erronés se cachent exactement comme zéro erreur. Connaître le nombre d'erreurs ne localiserait rien : sous l'hypothèse d'une seule erreur, c'est la différence entre les syndromes — les résultats du calcul de contrôle selon la position erronée — qui désigne la position fautive, et ici ils sont tous identiques. C'est pourquoi le tableau précédent s'arrêtait à `d_min = 2` pour ce code : avec une distance minimale de 2, on détecte une erreur mais on n'en corrige aucune.

Pour aller plus loin, il faut une redondance plus riche qu'un seul bit : c'est l'objet du code de Hamming.
## 4. Le code de Hamming (7,4) — et le décodage par syndrome

Le **code de Hamming (7,4)** encode 4 bits de message en un mot de 7 bits (3 bits de redondance). Sa distance minimale est **3** → il **corrige 1 erreur**. L'astuce est le **décodage par syndrome** :

- on définit une matrice de contrôle `H` (3×7) dont les 7 colonnes sont les 7 vecteurs **non nuls** de GF(2)³ ;
- le **syndrome** d'un mot reçu `r` est `s = H·r` ;
- si `s = 0`, aucune erreur (détectable) ;
- sinon, `s` est **égale à une colonne de H** — et celle-ci **pointe la position en erreur**. On la bascule, et le mot est corrigé.

C'est le « le syndrome pointe la colonne en erreur » de l'énoncé.

In [5]:
# Matrice generatrice G = [I4 | P] (systematique : les 4 bits de message sont en tete)
G = np.array([
    [1, 0, 0, 0, 1, 0, 1],
    [0, 1, 0, 0, 1, 1, 0],
    [0, 0, 1, 0, 1, 1, 1],
    [0, 0, 0, 1, 0, 1, 1],
], dtype=int)
# Matrice de controle H = [P^T | I3]  (les 7 colonnes = les 7 vecteurs non nuls de GF(2)^3)
H = np.array([
    [1, 1, 1, 0, 1, 0, 0],
    [0, 1, 1, 1, 0, 1, 0],
    [1, 0, 1, 1, 0, 0, 1],
], dtype=int)

# coherence : H . G^T = 0  (chaque codeur est dans le noyau de H)
assert np.all((H @ G.T) % 2 == 0), "H et G ne sont pas coherents"

# les 16 codeurs : tous les m . G pour m dans {0,1}^4
codewords = np.array([(m @ G) % 2 for m in
    (np.array([a, b, c, d]) for a in (0, 1) for b in (0, 1)
     for c in (0, 1) for d in (0, 1))])
print(f"nombre de codeurs : {len(codewords)}  (distincts : {len(np.unique(codewords, axis=0))})")
print(f"distance minimale : {min_distance(codewords)}")
assert len(np.unique(codewords, axis=0)) == 16
assert min_distance(codewords) == 3
assert np.unique(H, axis=0).shape[1] == 7, "les 7 colonnes de H doivent etre distinctes"
print("OK : 16 codeurs distincts, d_min = 3, 7 colonnes de H distinctes")

nombre de codeurs : 16  (distincts : 16)
distance minimale : 3
OK : 16 codeurs distincts, d_min = 3, 7 colonnes de H distinctes


In [6]:
def syndrome(r: np.ndarray) -> np.ndarray:
    """Syndrome d'un mot recu : s = H . r (dans GF(2))."""
    return (H @ r) % 2

def decode(r: np.ndarray) -> tuple:
    """Decode par syndrome : corrige 0 ou 1 erreur. Retourne (mot_corrige, position)."""
    s = syndrome(r)
    if not np.any(s):
        return r, None                                  # pas d'erreur detectable
    pos = int(np.where((H == s[:, None]).all(axis=0))[0][0])   # colonne de H = syndrome
    fixed = r.copy()
    fixed[pos] = 1 - fixed[pos]                         # bascule le bit en erreur
    return fixed, pos

# prise d'un codeur, injection d'UNE erreur, decodage
c = codewords[5]
noisy = c.copy(); noisy[3] = 1 - noisy[3]               # erreur en position 3
fixed, pos = decode(noisy)
print(f"codeur            = {c.tolist()}")
print(f"recu (erreur pos 3) = {noisy.tolist()}")
print(f"syndrome          = {syndrome(noisy).tolist()}  = colonne {pos} de H")
print(f"corrige           = {fixed.tolist()}   (retrouve le codeur ? {np.array_equal(fixed, c)})")
assert pos == 3 and np.array_equal(fixed, c)
assert np.all(syndrome(c) == 0), "le syndrome d'un codeur valide est nul"
print("OK : le syndrome pointe la colonne en erreur, le decodage restaure le codeur")

codeur            = [0, 1, 0, 1, 1, 0, 1]
recu (erreur pos 3) = [0, 1, 0, 0, 1, 0, 1]
syndrome          = [0, 1, 1]  = colonne 3 de H
corrige           = [0, 1, 0, 1, 1, 0, 1]   (retrouve le codeur ? True)
OK : le syndrome pointe la colonne en erreur, le decodage restaure le codeur


### Ce que montrent ces deux sorties

La première sortie **compte** ; la seconde **localise**. Ensemble, elles disent ce que le code de Hamming a de plus que le code de parité.

- **Le code a 16 codeurs, tous distincts, et sa distance minimale vaut 3.** L'assert `H · Gᵀ = 0` garantit que chaque mot produit par `G` appartient au noyau de `H` — c'est la cohérence du code — et la sortie note que les 7 colonnes de `H` sont distinctes, condition nécessaire pour que le syndrome puisse désigner une position unique.
- **Le syndrome est une position, pas un drapeau.** Sur le mot reçu `[0, 1, 0, 0, 1, 0, 1]`, le syndrome vaut `[0, 1, 1]`, et la sortie dit explicitement qu'il s'agit de la **colonne 3** de `H` — c'est-à-dire la position 3, celle qui avait été basculée. Le vecteur `[0, 1, 1]` n'est pas un code d'erreur abstrait : c'est l'adresse de l'erreur dans la table des colonnes.
- **La correction referme la boucle.** Le mot corrigé `[0, 1, 0, 1, 1, 0, 1]` est retrouvé bit à bit, et la sortie le prouve : `(retrouve le codeur ? True)`.

Ce que ces sorties **ne** prouvent pas : elles portent sur **une** erreur, à **une** position, dans **un** code. Elles ne montrent pas que le décodeur gère deux erreurs simultanées — le syndrome d'une double erreur peut être une colonne de `H` sans que l'erreur soit localisée, et le code présenterait alors une correction fausse. Cette limite tient à `d_min = 3` : au-delà d'une erreur, la correction n'est plus garantie. C'est le sens exact de la formule `⌊(d_min − 1) / 2⌋` vue plus haut, et c'est aussi ce qui rend la section suivante nécessaire : montrer que, pour une seule erreur, on ne peut pas faire mieux que remplir exactement l'espace.
## 5. La borne de Hamming — pourquoi le (7,4) est *parfait*

La **borne de Hamming** borne le nombre de codeurs d'un code qui corrige `t` erreurs. Si chaque codeur « possède » sa **sphère** de rayon `t` (tous les mots à distance ≤ `t`), ces sphères doivent être **disjointes** (sinon un mot serait à distance ≤ `t` de deux codeurs → ambiguïté). D'où :

`|C| × V ≤ 2ⁿ`,  où `V = Σ_{i=0}^{t} C(n, i)` est le volume de la sphère.

Le code (7,4) a `t = 1`, `n = 7`, `|C| = 16`. Le volume d'une sphère de rayon 1 vaut `V = C(7,0) + C(7,1) = 1 + 7 = 8`. Le produit `16 × 8 = 128 = 2⁷` : les sphères **pavent exactement** tout l'espace de 128 mots. Le code est dit **parfait** — aucune redondance n'est gaspillée.

In [7]:
from math import comb

n, k, t = 7, 4, 1
V = sum(comb(n, i) for i in range(t + 1))     # volume de la sphere de rayon t
M = 2 ** k                                     # nombre de codeurs
print(f"sphere de rayon {t}  : V = {V}   (= C(7,0) + C(7,1) = 1 + 7)")
print(f"nombre de codeurs    : M = {M}")
print(f"borne  : M x V = {M * V}   ;   taille de l'espace 2^n = {2 ** n}")
assert M * V == 2 ** n, "les spheres pavent exactement l'espace -> code PARFAIT"
print("OK : 16 x 8 = 128 = 2^7  ->  le (7,4) est PARFAIT (pavage exact, aucune place perdue)")

sphere de rayon 1  : V = 8   (= C(7,0) + C(7,1) = 1 + 7)
nombre de codeurs    : M = 16
borne  : M x V = 128   ;   taille de l'espace 2^n = 128
OK : 16 x 8 = 128 = 2^7  ->  le (7,4) est PARFAIT (pavage exact, aucune place perdue)


## Lecture

Cinq faits, tous **assertés** par le calcul :

1. **La distance de Hamming** est la brique de base : elle compte les positions où deux mots diffèrent.
2. **La distance minimale `d_min` achète** : `d_min − 1` erreurs détectées, `⌊(d_min−1)/2⌋` corrigées. Le code de parité (`d_min=2`) détecte 1 mais ne localise pas ; le (7,4) (`d_min=3`) corrige 1.
3. **Le code de parité ne corrige pas** : la parité ne répond qu'à « pair ou impair », et deux erreurs se cachent comme zéro erreur.
4. **Le syndrome pointe la colonne en erreur** : `s = H·r` égale la colonne de H qui indexe le bit fautif — c'est ce qui rend le (7,4) **corrigeable**, et non seulement détectable.
5. **Le (7,4) est parfait** : ses 16 sphères de rayon 1 pavent exactement les 128 mots de l'espace (`16 × 8 = 128 = 2⁷`).

**Ce que prouve la dernière mesure.** La sortie du code de la borne donne le volume d'une sphère de rayon 1 (`V = 8`), le nombre de codeurs (`M = 16`) et le produit (`M × V = 128`), puis le compare à la taille de l'espace (`2^n = 128`). L'égalité est **exacte**, pas approchée, et l'assert qui suit le calcul (`M * V == 2 ** n`) est la machine qui refuse tout arrondi. À elle seule, cette égalité ne dit rien des sphères : elle compare deux produits. Elle dit que les 16 sphères de rayon 1 remplissent l'espace **sans trou** une fois ajouté ce qu'une sortie antérieure a mesuré, la distance minimale 3 du code : deux codeurs distincts sont alors à au moins 3 positions l'un de l'autre, donc leurs sphères de rayon 1 sont disjointes, **sans recouvrement**. Seize sphères deux à deux disjointes de 8 mots chacune occupent 128 mots distincts : il n'y a ni trou ni place prise deux fois.

Ce que cette mesure **ne** prouve pas : que le (7,4) soit le meilleur code possible, ni qu'il soit pratique. Elle prouve l'**absence de gaspillage** pour la correction d'une seule erreur — on ne peut pas faire mieux que remplir exactement l'espace — et non l'optimalité absolue : un code qui corrige deux erreurs paie un volume de sphère plus grand, donc porte moins de codeurs. « Parfait » veut dire « sans reste », pas « sans compromis ».

**La suite.** Le carnet 02 monte en complexité : codes cycliques, Reed-Solomon, BCH (les codes des disques, QR, 5G). Le carnet 03 relie à Shannon : le théorème de codage, approcher la capacité, LDPC/turbo.

## Vérifiez votre compréhension

Trois prolongements. Chaque cellule code est un stub à compléter (pas d'erreur volontaire : le carnet s'exécute de bout en bout).


### Exercice 1 — Détecteur de parité

Écrire `detecteur_parite(codeword)` qui renvoie `True` si le mot a un nombre **impair** de 1 (une erreur détectable), `False` sinon.

In [8]:
def detecteur_parite(codeword: np.ndarray) -> bool:
    """Renvoie True si le nombre de 1 est impair (erreur detectable), False sinon.

    # TODO etudiant : sommer les bits, tester la parite (mod 2).
    """
    result = None  # TODO etudiant
    return result

### Exercice 2 — Distance minimale

Écrire `d_min(codeurs)` qui renvoie la distance minimale d'un code : la plus petite distance de Hamming entre deux codeurs distincts.

In [9]:
def d_min(codeurs: np.ndarray) -> int:
    """Distance minimale : min des distances de Hamming entre paires distinctes.

    # TODO etudiant : parcourir les paires (i, j) avec i < j, garder le minimum.
    """
    result = None  # TODO etudiant
    return result

### Exercice 3 — Position d'erreur par syndrome

Écrire `position_erreur(r)` qui, donné un mot reçu `r`, renvoie l'indice du bit en erreur (en trouvant la colonne de `H` égale au syndrome), ou `None` si le syndrome est nul.

In [10]:
def position_erreur(r: np.ndarray):
    """Indice du bit en erreur (colonne de H egale au syndrome), ou None si syndrome nul.

    # Indice : le syndrome vaut (H @ r) % 2 ; le comparer a chaque colonne de H.
    # TODO etudiant
    """
    result = None  # TODO etudiant
    return result

## À retenir

Ce carnet a transformé « le canal est bruité » en **objets calculables** : une distance, une distance minimale qui achète une capacité, un code de parité, un code de Hamming qui **corrige** par syndrome, et une borne qui explique pourquoi le (7,4) est parfait. Tous les résultats sont **assertés** et **reproductibles en CPU**.


## Pour aller plus loin

La suite de la série pousse vers les codes réels (Reed-Solomon, BCH) et vers la capacité de Shannon.

- David J. C. MacKay, *Information Theory, Inference, and Learning Algorithms*, Cambridge
  University Press, 2003 — le référentiel abordable sur les codes correcteurs d'erreurs.
- Yury Polyanskiy, Yihong Wu, *Information Theory — From Coding to Learning*, Cambridge
  University Press — la copie consultée en est la version prépublication du 16 août 2024
  (brouillon des auteurs, antérieur à l'édition imprimée) ; le traitement moderne,
  du codage à l'apprentissage.
- Dans la série : [Carnet 01 — Entropie et information mutuelle](InformationTheory-01-Entropie-Et-Information-Mutuelle.ipynb)
    (ce que la capacité `C` mesure) et [Carnet 08 — Théorie de l'information des réseaux](InformationTheory-08-Theorie-De-L-Information-Des-Reseaux.ipynb)
    (plusieurs émetteurs, plusieurs récepteurs).
